# EfficientNet: Compound Scaling for Optimal Efficiency

**The Problem with Scaling**: MobileNet gave us efficient building blocks (depthwise separable convolutions), but how do we scale networks up or down optimally?

Traditional approaches scale **one dimension at a time**:
- **Depth scaling**: Add more layers (ResNet-18 → ResNet-50 → ResNet-101)
- **Width scaling**: Add more channels per layer (MobileNet-1.0 → MobileNet-1.4)
- **Resolution scaling**: Use larger input images (224×224 → 299×299)

But this is **suboptimal**! A network that processes 2× larger images should have:
- More depth (to capture more complex patterns)
- More width (to capture more features)
- Not just larger inputs!

**EfficientNet's Innovation**: Use **Neural Architecture Search (NAS)** to find an efficient base architecture, then scale **all three dimensions together** using a **compound scaling** method that balances depth, width, and resolution with fixed ratios.

**Result**: State-of-the-art accuracy with up to 10× better efficiency than previous CNNs.

This notebook:
1. Shows why single-dimension scaling is inefficient
2. Explains compound scaling mathematically
3. Implements EfficientNet-B0 with MBConv blocks and Squeeze-and-Excitation
4. Demonstrates scaling to larger models (B1, B2, ...)
5. Compares efficiency vs accuracy trade-offs

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm
import math

## 1. The Scaling Problem: Why Single-Dimension Scaling is Suboptimal

When we want a more powerful network, we have three knobs to turn:

### Depth Scaling (d)
Add more layers: ResNet-18 → ResNet-50 → ResNet-101
- ✓ Captures richer, more complex features
- ✗ Harder to train (gradient vanishing)
- ✗ Diminishing returns

### Width Scaling (w)
Add more channels per layer: 64 → 128 → 256
- ✓ Captures more fine-grained features
- ✗ Saturates quickly at very wide networks

### Resolution Scaling (r)
Use larger input images: 224×224 → 299×299 → 331×331
- ✓ Captures finer details
- ✗ Network needs more capacity to process larger inputs

**Intuition**: If you give a network 2× larger images (4× more pixels), it needs:
- More depth to have larger receptive fields
- More width to capture more patterns
- Not just the same architecture with bigger inputs!

**Problem**: How do we balance these three dimensions?

## 2. Compound Scaling: The EfficientNet Solution

EfficientNet scales **all three dimensions together** with a **compound coefficient φ**:

$$
\begin{align}
\text{depth:} \quad d &= \alpha^\phi \\
\text{width:} \quad w &= \beta^\phi \\
\text{resolution:} \quad r &= \gamma^\phi
\end{align}
$$

Where:
- **α, β, γ** are constants found by grid search (α=1.2, β=1.1, γ=1.15 for EfficientNet)
- **φ** is the compound coefficient that controls overall scaling
- Constraint: **α · β² · γ² ≈ 2** (approximately doubles FLOPs for each φ increment)

**Why β² and γ²?**
- Doubling width (channels) → 2× more filters → **4× FLOPs** (both input and output channels double)
- Doubling resolution → 4× more pixels → **4× FLOPs**
- Doubling depth → 2× more layers → **2× FLOPs**

So: FLOPS ∝ d · w² · r²

### EfficientNet Family
- **EfficientNet-B0**: Baseline found by NAS (φ=0)
- **EfficientNet-B1**: φ=1 (2× FLOPs)
- **EfficientNet-B2**: φ=2 (4× FLOPs)
- ...
- **EfficientNet-B7**: φ=7 (128× FLOPs)

Each model scales depth, width, and resolution together!

## 3. Building Blocks: MBConv with Squeeze-and-Excitation

EfficientNet uses **MBConv blocks** (Mobile Inverted Bottleneck Convolution), which are:
- Depthwise separable convolutions (from MobileNet)
- Inverted bottleneck structure (expand → filter → squeeze)
- **Squeeze-and-Excitation (SE)** for channel attention

### MBConv Block Structure
1. **Expansion**: 1×1 conv to expand channels (e.g., 32 → 192, expansion ratio = 6)
2. **Depthwise**: 3×3 or 5×5 depthwise conv (spatial filtering)
3. **Squeeze-and-Excitation**: Channel attention (adaptive feature recalibration)
4. **Projection**: 1×1 conv to project back (e.g., 192 → 32)
5. **Skip connection**: If input/output same shape

### Squeeze-and-Excitation (SE)
Adaptively recalibrates channel-wise feature responses:
1. **Squeeze**: Global average pooling (H×W → 1×1)
2. **Excitation**: Two FC layers (reduce → expand) with sigmoid
3. **Scale**: Multiply original features by attention weights

**Intuition**: "Which channels are important for this input?" Learn to emphasize useful features and suppress less useful ones.

Now build the MBConv block that EfficientNet scales across stages.

In [ ]:
class SqueezeExcitation(nn.Module):
    """Squeeze-and-Excitation block for channel attention."""
    
    def __init__(self, in_channels, reduced_dim):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),  # Squeeze: H×W → 1×1
            nn.Conv2d(in_channels, reduced_dim, 1),  # Reduce
            nn.SiLU(inplace=True),  # Swish activation
            nn.Conv2d(reduced_dim, in_channels, 1),  # Expand
            nn.Sigmoid()  # Attention weights in [0, 1]
        )
    
    def forward(self, x):
        return x * self.se(x)  # Scale features by attention

Define the MBConv block that EfficientNet scales across stages.

In [ ]:
class MBConvBlock(nn.Module):
    """Mobile Inverted Bottleneck Convolution (MBConv) block.
    
    Args:
        in_channels: Input channels
        out_channels: Output channels
        expand_ratio: Expansion ratio (typically 1, 4, or 6)
        kernel_size: Depthwise conv kernel (3 or 5)
        stride: Stride for depthwise conv
        se_ratio: Squeeze-excitation ratio (0.25 means reduce to 1/4 channels)
    """
    
    def __init__(self, in_channels, out_channels, expand_ratio, 
                 kernel_size, stride, se_ratio=0.25):
        super().__init__()
        self.use_residual = (stride == 1 and in_channels == out_channels)
        hidden_dim = in_channels * expand_ratio
        self.expand = expand_ratio != 1
        
        layers = []
        
        # 1. Expansion (if needed)
        if self.expand:
            layers.extend([
                nn.Conv2d(in_channels, hidden_dim, 1, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.SiLU(inplace=True)  # Swish activation
            ])
        
        # 2. Depthwise convolution
        layers.extend([
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size, stride,
                     padding=kernel_size // 2, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.SiLU(inplace=True)
        ])
        
        self.conv = nn.Sequential(*layers)
        
        # 3. Squeeze-and-Excitation
        reduced_dim = max(1, int(in_channels * se_ratio))
        self.se = SqueezeExcitation(hidden_dim, reduced_dim)
        
        # 4. Projection
        self.project = nn.Sequential(
            nn.Conv2d(hidden_dim, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
    
    def forward(self, x):
        identity = x
        x = self.conv(x)
        x = self.se(x)
        x = self.project(x)
        
        if self.use_residual:
            x = x + identity
        
        return x

## 4. EfficientNet-B0 Architecture

The base architecture (B0) was found using Neural Architecture Search (NAS).

**Architecture**:
- Stage 1: MBConv1 (k3×3, expansion=1)
- Stage 2: MBConv6 (k3×3, expansion=6)
- Stage 3: MBConv6 (k5×5, expansion=6)
- Stage 4-7: MBConv6 (varying kernel sizes and channels)

Each stage progressively increases channels and decreases spatial resolution.

In [ ]:
class EfficientNet(nn.Module):
    """EfficientNet with compound scaling.
    
    Args:
        width_mult: Width multiplier (scales channels)
        depth_mult: Depth multiplier (scales layers)
        num_classes: Number of output classes
    """
    
    def __init__(self, width_mult=1.0, depth_mult=1.0, num_classes=10):
        super().__init__()
        
        # EfficientNet-B0 configuration
        # [expand_ratio, channels, num_layers, stride, kernel_size]
        config = [
            [1,  16,  1, 1, 3],  # Stage 1
            [6,  24,  2, 2, 3],  # Stage 2
            [6,  40,  2, 2, 5],  # Stage 3
            [6,  80,  3, 2, 3],  # Stage 4
            [6, 112,  3, 1, 5],  # Stage 5
            [6, 192,  4, 2, 5],  # Stage 6
            [6, 320,  1, 1, 3],  # Stage 7
        ]
        
        # Initial stem
        out_channels = self._scale_width(32, width_mult)
        self.stem = nn.Sequential(
            nn.Conv2d(3, out_channels, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True)
        )
        
        # Build MBConv stages
        layers = []
        in_channels = out_channels
        
        for expand_ratio, channels, num_layers, stride, kernel_size in config:
            out_channels = self._scale_width(channels, width_mult)
            num_layers = self._scale_depth(num_layers, depth_mult)
            
            for i in range(num_layers):
                layers.append(
                    MBConvBlock(
                        in_channels if i == 0 else out_channels,
                        out_channels,
                        expand_ratio,
                        kernel_size,
                        stride if i == 0 else 1
                    )
                )
            in_channels = out_channels
        
        self.blocks = nn.Sequential(*layers)
        
        # Head
        final_channels = self._scale_width(1280, width_mult)
        self.head = nn.Sequential(
            nn.Conv2d(in_channels, final_channels, 1, bias=False),
            nn.BatchNorm2d(final_channels),
            nn.SiLU(inplace=True),
            nn.AdaptiveAvgPool2d(1)
        )
        
        self.classifier = nn.Linear(final_channels, num_classes)
    
    def _scale_width(self, channels, width_mult):
        """Scale channel count based on width multiplier."""
        channels = int(channels * width_mult)
        # Round to nearest multiple of 8 for hardware efficiency
        return int((channels + 4) // 8) * 8
    
    def _scale_depth(self, num_layers, depth_mult):
        """Scale layer count based on depth multiplier."""
        return int(math.ceil(num_layers * depth_mult))
    
    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head(x)
        x = x.flatten(1)
        x = self.classifier(x)
        return x

## 5. Compound Scaling in Action

Let's see how the scaling formula works for different EfficientNet variants:

**Formula**: d = α^φ, w = β^φ, r = γ^φ (where α=1.2, β=1.1, γ=1.15)

We'll demonstrate the scaling by creating different model sizes.

In [ ]:
def create_efficientnet(variant='B0', num_classes=10):
    """Create EfficientNet model with compound scaling.
    
    Args:
        variant: 'B0', 'B1', 'B2', etc.
        num_classes: Number of output classes
    """
    # Compound scaling coefficients
    # α = 1.2 (depth), β = 1.1 (width), γ = 1.15 (resolution)
    # φ is the compound coefficient
    
    configs = {
        'B0': (1.0, 1.0, 224),  # φ=0: baseline
        'B1': (1.2, 1.1, 256),  # φ=1: α^1, β^1, γ^1
        'B2': (1.4, 1.2, 288),  # φ=2: α^2, β^2, γ^2
    }
    
    depth_mult, width_mult, resolution = configs.get(variant, configs['B0'])
    
    return EfficientNet(
        width_mult=width_mult,
        depth_mult=depth_mult,
        num_classes=num_classes
    ), resolution

## 6. Visualizing Compound Scaling

Let's see how parameters and FLOPs scale across variants.

In [ ]:
def count_parameters(model):
    """Count trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Compare model sizes
variants = ['B0', 'B1', 'B2']
params = []

for variant in variants:
    model, _ = create_efficientnet(variant)
    num_params = count_parameters(model)
    params.append(num_params / 1e6)  # Convert to millions
    print(f"EfficientNet-{variant}: {num_params/1e6:.2f}M parameters")

# Visualize scaling
plt.figure(figsize=(8, 5))
plt.bar(variants, params, color=['#3498db', '#e74c3c', '#2ecc71'])
plt.xlabel('EfficientNet Variant', fontsize=12)
plt.ylabel('Parameters (Millions)', fontsize=12)
plt.title('Compound Scaling: Parameter Growth', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, (variant, param) in enumerate(zip(variants, params)):
    plt.text(i, param + 0.1, f'{param:.2f}M', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print(f"\n📊 B1 has {params[1]/params[0]:.2f}× more parameters than B0")
print(f"📊 B2 has {params[2]/params[0]:.2f}× more parameters than B0")

## 7. Training EfficientNet-B0 on CIFAR-10

We'll train EfficientNet-B0 (the smallest variant) on CIFAR-10 to demonstrate its efficiency.

In [ ]:
# Data preparation
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                             download=True, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                            download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Set up the model, optimizer, and training configuration.

In [ ]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 
                     'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

# Create model (B0 for CIFAR-10's 32×32 images)
model, _ = create_efficientnet('B0', num_classes=10)
model = model.to(device)

print(f"\nModel: EfficientNet-B0")
print(f"Parameters: {count_parameters(model)/1e6:.2f}M")

# Optimizer and scheduler
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

Define training and evaluation functions.

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc='Training', leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': f'{loss.item():.3f}', 
                         'acc': f'{100.*correct/total:.2f}%'})
    
    return total_loss / len(loader), 100. * correct / total

def evaluate(model, loader, criterion, device):
    """Evaluate model on validation/test set."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Evaluating', leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return total_loss / len(loader), 100. * correct / total

Train the model for 20 epochs.

In [ ]:
num_epochs = 20
train_losses, train_accs = [], []
test_losses, test_accs = [], []

print("\nTraining EfficientNet-B0...\n")

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    scheduler.step()
    
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    test_losses.append(test_loss)
    test_accs.append(test_acc)
    
    print(f"Epoch {epoch+1:2d}/{num_epochs} | "
          f"Train Loss: {train_loss:.3f}, Train Acc: {train_acc:.2f}% | "
          f"Test Loss: {test_loss:.3f}, Test Acc: {test_acc:.2f}%")

print(f"\n✅ Final Test Accuracy: {test_accs[-1]:.2f}%")

## 8. Training Results

Visualize training curves to see convergence.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax1.plot(train_losses, label='Train Loss', linewidth=2, color='#3498db')
ax1.plot(test_losses, label='Test Loss', linewidth=2, color='#e74c3c')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('EfficientNet-B0: Loss Curves', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Accuracy curves
ax2.plot(train_accs, label='Train Accuracy', linewidth=2, color='#3498db')
ax2.plot(test_accs, label='Test Accuracy', linewidth=2, color='#e74c3c')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('EfficientNet-B0: Accuracy Curves', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 EfficientNet-B0 achieved {test_accs[-1]:.2f}% test accuracy")
print(f"📊 With only {count_parameters(model)/1e6:.2f}M parameters!")

## 9. Comparing Architectures: Efficiency vs Accuracy

Let's visualize how EfficientNet compares to previous architectures in the efficiency-accuracy landscape.

**Key Insight**: EfficientNet achieves better accuracy with fewer parameters and FLOPs than ResNet, VGG, and even some Inception variants.

In [ ]:
# Architecture comparison (approximate values for ImageNet)
architectures = {
    'AlexNet': (60, 57.0),      # (params in M, top-1 accuracy %)
    'VGG-16': (138, 71.5),
    'ResNet-50': (25.6, 76.0),
    'ResNet-152': (60.2, 77.8),
    'Inception-v3': (23.8, 77.5),
    'MobileNet-v1': (4.2, 70.6),
    'EfficientNet-B0': (5.3, 77.3),
    'EfficientNet-B1': (7.8, 79.2),
    'EfficientNet-B7': (66.0, 84.4),
}

# Extract data
names = list(architectures.keys())
params = [v[0] for v in architectures.values()]
accs = [v[1] for v in architectures.values()]

# Color by family
colors = ['#95a5a6', '#95a5a6', '#3498db', '#3498db', '#9b59b6', 
          '#e67e22', '#2ecc71', '#2ecc71', '#2ecc71']

plt.figure(figsize=(12, 7))
plt.scatter(params, accs, s=200, c=colors, alpha=0.7, edgecolors='black', linewidth=1.5)

# Add labels
for i, name in enumerate(names):
    offset_x = 3 if 'EfficientNet' in name else -3
    offset_y = 0.3 if i % 2 == 0 else -0.5
    plt.annotate(name, (params[i], accs[i]), 
                xytext=(offset_x, offset_y), textcoords='offset points',
                fontsize=10, fontweight='bold' if 'EfficientNet' in name else 'normal')

plt.xlabel('Parameters (Millions)', fontsize=13)
plt.ylabel('ImageNet Top-1 Accuracy (%)', fontsize=13)
plt.title('CNN Architectures: Efficiency vs Accuracy Trade-off', 
         fontsize=15, fontweight='bold')
plt.grid(alpha=0.3)

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#95a5a6', label='Early CNNs'),
    Patch(facecolor='#3498db', label='ResNet'),
    Patch(facecolor='#9b59b6', label='Inception'),
    Patch(facecolor='#e67e22', label='MobileNet'),
    Patch(facecolor='#2ecc71', label='EfficientNet')
]
plt.legend(handles=legend_elements, loc='lower right', fontsize=11)

plt.tight_layout()
plt.show()

print("\n🎯 EfficientNet-B0 achieves ResNet-50 level accuracy with 5× fewer parameters!")
print("🎯 EfficientNet-B7 achieves 84.4% accuracy (best on this chart) with similar params to VGG-16")

## 10. Key Innovations Recap

### Why EfficientNet Matters

**1. Compound Scaling** 🎯
- Scales depth, width, and resolution together
- Balances all three dimensions for optimal efficiency
- Simple formula: d=α^φ, w=β^φ, r=γ^φ

**2. Neural Architecture Search** 🔍
- Uses NAS to find optimal base architecture (B0)
- Discovers MBConv blocks with optimal kernel sizes
- Human intuition + automated search = better architectures

**3. Building on MobileNet** 📱
- Uses depthwise separable convolutions (efficient)
- Adds Squeeze-and-Excitation for channel attention
- Inverted bottleneck design (expand → filter → project)

**4. State-of-the-Art Efficiency** ⚡
- EfficientNet-B0: 77.3% accuracy, 5.3M params
- EfficientNet-B7: 84.4% accuracy, 66M params
- Up to 10× better efficiency than previous CNNs

### The Efficiency Equation

$$
\text{Efficiency} = \frac{\text{Accuracy}}{\text{Parameters} \times \text{FLOPs}}
$$

EfficientNet maximizes this ratio through compound scaling!

## 11. When to Use EfficientNet

**Use EfficientNet when**:
- ✅ You need efficient inference (mobile, edge devices)
- ✅ You want to scale a model family (B0 → B7)
- ✅ You care about parameter efficiency
- ✅ You need strong transfer learning features

**Consider alternatives when**:
- ❌ You need the absolute simplest architecture (use ResNet)
- ❌ You're doing research requiring full interpretability
- ❌ You have unlimited compute (use Vision Transformers)
- ❌ You need the fastest training time (SE blocks add overhead)

### Beyond CNNs: What's Next?

EfficientNet represents the **peak of CNN efficiency**, but:
- **Vision Transformers (ViT)**: Self-attention for images
- **ConvNeXt**: Modernizing CNNs with Transformer ideas
- **EfficientNetV2**: Even faster training with progressive learning

**The CNN journey**:
- LeNet → AlexNet: Depth matters
- VGG: Uniform small filters
- Inception: Multi-scale features
- ResNet: Skip connections enable depth
- MobileNet: Depthwise separable for efficiency
- **EfficientNet: Compound scaling for optimal balance** ✨

## Summary

**EfficientNet's breakthrough**: Compound scaling that balances depth, width, and resolution.

**Key concepts**:
1. **Single-dimension scaling** is suboptimal (depth only, width only, or resolution only)
2. **Compound scaling formula**: d=α^φ, w=β^φ, r=γ^φ with constraint α·β²·γ²≈2
3. **MBConv blocks**: Mobile inverted bottleneck + Squeeze-and-Excitation
4. **Neural Architecture Search**: Find optimal base architecture
5. **Efficiency**: 10× better parameters/accuracy trade-off than previous CNNs

**Impact**: EfficientNet family (B0-B7) provides scalable, efficient models for any compute budget. Compound scaling principle applies beyond just CNNs - it's a general principle for scaling neural networks efficiently.

**What we learned**:
- Architecture design isn't just about accuracy - **efficiency matters**
- Scaling all dimensions together > scaling one dimension
- Automated search (NAS) can discover better designs than human intuition alone
- The journey from LeNet (1998) to EfficientNet (2019) shows systematic progress in CNN design

This completes the CNN evolution story. Next frontier: **Vision Transformers** that use self-attention instead of convolutions! 🚀